In [1]:
################
# Tabular ARGN #
################

## Authors:             Ricardo Zambrano
## email:               rzambrano@gmail.com
## Date:               2026-03-14

## reference:           "TabularARGN: A flexible and Efficient Auto-Regressive Framework for Generating high-Fidelity Synthetic Data"
##                      Tiwald et al. (2025)
##                      https://arxiv.org/pdf/2501.12012

######################
# Tabular ARGN       #
######################

## Author:             Ricardo Zambrano
## Email:              rzambrano@gmail.com
## Date:               2026-03-14
## Reference:          "TabularARGN: A Flexible and Efficient Auto-Regressive Framework
##                      for Generating High-Fidelity Synthetic Data"
##                      Tiwald et al. (2025)
##                      https://arxiv.org/abs/[arxiv-id]  # fill this in

#################
# Session Setup #
#################

# ── Standard Library ──────────────────────────────────────────────────────────
import os
import random
from pathlib import Path

from typing import Protocol

from datetime import date, datetime

# ── Numerical & Data ──────────────────────────────────────────────────────────
import numpy as np
import polars as pl
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Experiment Tracking ───────────────────────────────────────────────────────
import wandb
import mlflow
from omegaconf import DictConfig, OmegaConf
import hydra

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import f1_score, roc_auc_score
import optuna

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ── Hardcoded Variables ────────────────────────────────────────────────────────

# -- None --


c:\Users\rzamb\Documents\tabular_nn\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Working Directory ──────────────────────────────────────────────────────────

current_dir = Path.cwd().resolve()
print(current_dir)

target_dir = Path(r"c:\Users\rzamb\Documents\tabular_nn").resolve()

if current_dir != target_dir:

    # Get the path object for two levels up
    two_levels_up = current_dir.parents[0]

    # Change the working directory to the new path
    os.chdir(two_levels_up)

# Optional: Print new working directory to confirm the change
print(f"New working directory: {Path.cwd()}")

C:\Users\rzamb\Documents\tabular_nn\notebooks
New working directory: C:\Users\rzamb\Documents\tabular_nn


In [3]:
####################
# Helper Functions #
####################

# -- None --

In [4]:
####################
# Data Loading     #
####################

insurance_data = pd.read_csv("./data/raw/insurance/insurance.csv")

In [5]:
insurance_data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [6]:
insurance_data.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [7]:
insurance_data.dtypes

age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

In [8]:
beijing_data = pd.read_csv("./data/raw/beijing/PRSA_Data_Dingling_20130301-20170228.csv")

In [9]:
beijing_data.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,Dingling
1,2,2013,3,1,1,7.0,7.0,3.0,NaN,200.0,80.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Dingling
2,3,2013,3,1,2,5.0,5.0,3.0,2.0,200.0,79.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Dingling
3,4,2013,3,1,3,6.0,6.0,3.0,NaN,200.0,79.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Dingling
4,5,2013,3,1,4,5.0,5.0,3.0,NaN,200.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Dingling


In [10]:
beijing_data["sample_date"] = pd.to_datetime({
    'year': beijing_data['year'],
    'month': beijing_data['month'],
    'day': beijing_data['day']
})

In [11]:
beijing_data = beijing_data.drop(columns=["year", "month", "day"])

In [12]:
beijing_data.dtypes

No                      int64
hour                    int64
PM2.5                 float64
PM10                  float64
SO2                   float64
NO2                   float64
CO                    float64
O3                    float64
TEMP                  float64
PRES                  float64
DEWP                  float64
RAIN                  float64
wd                     object
WSPM                  float64
station                object
sample_date    datetime64[ns]
dtype: object

In [13]:
####################
# Data Wrangling   #
####################

In [14]:
# Introducing a null values in different formats

insurance_data.iloc[0,0] = pd.NA
insurance_data.iloc[1,0] = np.nan
insurance_data.iloc[2,1] = None

# Inserting an artificial person with a number of children disconnected   from the sequence 1->5
insurance_data.iloc[3,3] = 14

# Rounding BMI to get a BINNED strategy
insurance_data["bmi"] = insurance_data["bmi"].round(1)

In [35]:
import importlib
import src.utils.tabular_datasets as tabular_datasets
importlib.reload(tabular_datasets)
from src.utils.tabular_datasets import ArgnDataset

In [ ]:
import warnings
import logging

In [56]:
from src.utils.argn_encoder_decoder import BinDesign

In [57]:
one_col = BinDesign(1,[2,3])
another_col = BinDesign(2,[4,5,6])
test_bin_mapping_list = [one_col, another_col]
test_bin_mapping_dict = {"one_col_key" : BinDesign(1,[2,3]), "another_col_key" : BinDesign(2,[4,5,6])}

In [58]:
test_bin_mapping_list[0].n_bins

1

In [59]:
test_bin_mapping_dict["one_col_key"].n_bins

1

In [87]:
from src.utils.argn_encoder_decoder import decode_numerical_binned, decode_categorical, decode_numerical_discrete, decode_numerical_digit

In [88]:
decode_numerical_binned(
    insurance_table_test._table,
    insurance_table_test.numerical_binned_decoding_maps,
    [item[0] for item in insurance_table_test._numerical_binned_columns]
)


age,sex,bmi,children,smoker,region,charges_1,charges_2,charges_3,charges_4,charges_5,charges_6,charges_7,charges_8,charges_9,charges_10,charges_11,charges_12,charges_13
i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,1,27.874908,0,1,3,0,1,6,8,8,4,9,2,4,0,0,0,0
0,2,33.890143,1,0,2,0,0,1,7,2,5,5,5,2,3,0,0,0
11,0,33.046399,3,0,2,0,0,4,4,4,9,4,6,2,0,0,0,0
16,2,22.839463,14,0,1,0,2,1,9,8,4,4,7,0,6,1,0,0
15,2,28.931204,0,0,1,0,0,3,8,6,6,8,5,5,2,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
33,2,31.134505,3,0,1,0,1,0,6,0,0,5,4,8,3,0,0,0
1,1,32.014454,0,0,0,0,0,2,2,0,5,9,8,0,8,0,0,0
1,1,36.930065,0,0,2,0,0,1,6,2,9,8,3,3,5,0,0,0


In [90]:
decode_categorical(
    insurance_table_test._table,
    insurance_table_test.categorical_decoding_maps ,
    [item[0] for item in insurance_table_test._categorical_columns]
)

age,sex,bmi,children,smoker,region,charges_1,charges_2,charges_3,charges_4,charges_5,charges_6,charges_7,charges_8,charges_9,charges_10,charges_11,charges_12,charges_13
i64,str,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,"""female""",35,0,"""yes""","""southwest""",0,1,6,8,8,4,9,2,4,0,0,0,0
0,"""male""",71,1,"""no""","""southeast""",0,0,1,7,2,5,5,5,2,3,0,0,0
11,null,66,3,"""no""","""southeast""",0,0,4,4,4,9,4,6,2,0,0,0,0
16,"""male""",10,14,"""no""","""northwest""",0,2,1,9,8,4,4,7,0,6,1,0,0
15,"""male""",42,0,"""no""","""northwest""",0,0,3,8,6,6,8,5,5,2,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
33,"""male""",55,3,"""no""","""northwest""",0,1,0,6,0,0,5,4,8,3,0,0,0
1,"""female""",60,0,"""no""","""northeast""",0,0,2,2,0,5,9,8,0,8,0,0,0
1,"""female""",85,0,"""no""","""southeast""",0,0,1,6,2,9,8,3,3,5,0,0,0


In [95]:
decode_numerical_discrete(
    insurance_table_test._table,
    insurance_table_test.numerical_discrete_decoding_maps,
    [item[0] for item in insurance_table_test._numerical_discrete_columns]    
).head(10)

age,sex,bmi,children,smoker,region,charges_1,charges_2,charges_3,charges_4,charges_5,charges_6,charges_7,charges_8,charges_9,charges_10,charges_11,charges_12,charges_13
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
null,1,35,0,1,3,0,1,6,8,8,4,9,2,4,0,0,0,0
null,2,71,1,0,2,0,0,1,7,2,5,5,5,2,3,0,0,0
28,0,66,3,0,2,0,0,4,4,4,9,4,6,2,0,0,0,0
33,2,10,null,0,1,0,2,1,9,8,4,4,7,0,6,1,0,0
32,2,42,0,0,1,0,0,3,8,6,6,8,5,5,2,0,0,0
31,1,23,0,0,2,0,0,3,7,5,6,6,2,1,6,0,0,0
46,1,69,1,0,2,0,0,8,2,4,0,5,8,9,6,0,0,0
37,1,34,3,0,1,0,0,7,2,8,1,5,0,5,6,0,0,0
37,2,47,2,0,0,0,0,6,4,0,6,4,1,0,7,0,0,0


In [118]:
import re
col_name = 'charges'
curr_col_encoding = insurance_table_test.numerical_digit_encoding_maps[col_name]
all_columns_in_encoding = [item for item in insurance_table_test._table.columns if re.search(col_name, item, re.IGNORECASE)]
all_columns_in_encoding

['charges_1',
 'charges_2',
 'charges_3',
 'charges_4',
 'charges_5',
 'charges_6',
 'charges_7',
 'charges_8',
 'charges_9',
 'charges_10',
 'charges_11',
 'charges_12',
 'charges_13']

In [123]:
decode_numerical_digit(
        df_pl = insurance_table_test._table,
        digit_cols = [item[0] for item in insurance_table_test._numerical_digit_columns],
        digit_encodings = insurance_table_test.numerical_digit_encoding_maps)

age,sex,bmi,children,smoker,region,charges
i64,i64,i64,i64,i64,i64,f64
0,1,35,0,1,3,16884.924
0,2,71,1,0,2,1725.5523
11,0,66,3,0,2,4449.462
16,2,10,14,0,1,21984.47061
15,2,42,0,0,1,3866.8552
…,…,…,…,…,…,…
33,2,55,3,0,1,10600.5483
1,1,60,0,0,0,2205.9808
1,1,85,0,0,2,1629.8335


In [133]:
from datetime import date, time

# Creating the sample data
data = {
    "Date": [date(2026, 3, 23), date(2026, 1, 1)],
    "Time": [time(14, 30), time(9, 15)],
    "Datetime": pd.to_datetime(["2026-03-23 14:30:00", "2026-01-01 09:15:00"]),
    "Duration": pd.to_timedelta(["2 days 5 hours", "12 hours 30 min"]),
    "bool_test": [True, False]
}

testing_datetime_types_df = pd.DataFrame(data)

In [134]:
testing_datetime_types_df

,Date,Time,Datetime,Duration,bool_test
0,2026-03-23,14:30:00,2026-03-23 14:30:00,2 days 05:00:00,True
1,2026-01-01,09:15:00,2026-01-01 09:15:00,0 days 12:30:00,False


In [135]:
testing_datetime_types_pl = pl.from_pandas(testing_datetime_types_df)

In [136]:
testing_datetime_types_pl

Date,Time,Datetime,Duration,bool_test
date,time,datetime[ns],duration[ns],bool
2026-03-23,14:30:00,2026-03-23 14:30:00,2d 5h,true
2026-01-01,09:15:00,2026-01-01 09:15:00,12h 30m,false


In [137]:
testing_datetime_types_pl.dtypes

[Date,
 Time,
 Datetime(time_unit='ns', time_zone=None),
 Duration(time_unit='ns'),
 Boolean]

In [138]:
# pl.Date     YYYY-MM-DD
# pl.Datetime YYYY-MM-DD HH:MM:SS
# pl.Time     HH:MM:SS
# pl.Duration

In [139]:
isinstance(df_pl[col_name].dtype, pl.Datetime)

True

In [147]:
datetime_test = ArgnDataset(df_pl.to_pandas(), clip_cols = True)

C:\Users\rzamb\Documents\tabular_nn\src\utils\argn_encoder_decoder.py:418: UserWarning: get_n_bins() received an empty list as an argument...
  warnings.warn("get_n_bins() received an empty list as an argument...")


In [148]:
datetime_test._incompatible_columns

[]

In [149]:
datetime_test.dtypes

['Datetime', 'Time', 'Datetime', 'Duration', 'Boolean']

In [150]:
datetime_test._datetime_columns

[('Date', 0), ('Time', 1), ('Datetime', 2)]

In [151]:
datetime_test._duration_columns

[('Duration', 3)]

In [152]:
datetime_test.datetime_encoding_map

{'Date': Datetime, 'Time': Time, 'Datetime': Datetime}

In [153]:
datetime_test._table

Duration,bool_test,Date_year,Date_month,Date_day,Date_hour,Date_minute,Date_second,Date_millisecond,Time_hour,Time_minute,Time_second,Time_millisecond,Datetime_year,Datetime_month,Datetime_day,Datetime_hour,Datetime_minute,Datetime_second,Datetime_millisecond
i64,i8,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,1,0,1,1,0,0,0,0,1,1,0,0,0,1,1,1,1,0,0
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [154]:
df_pl

Date,Time,Datetime,Duration,bool_test
date,time,datetime[ns],duration[ns],bool
2026-03-23,14:30:00,2026-03-23 14:30:00,2d 5h,true
2026-01-01,09:15:00,2026-01-01 09:15:00,12h 30m,false


In [155]:
datetime_test.datetime_discretized_encoding_maps

{'Duration': {45000: 0, 190800: 1},
 'Date_year': {2026: 0},
 'Date_month': {1: 0, 3: 1},
 'Date_day': {1: 0, 23: 1},
 'Date_hour': {0: 0},
 'Date_minute': {0: 0},
 'Date_second': {0: 0},
 'Date_millisecond': {0: 0},
 'Time_hour': {9: 0, 14: 1},
 'Time_minute': {15: 0, 30: 1},
 'Time_second': {0: 0},
 'Time_millisecond': {0: 0},
 'Datetime_year': {2026: 0},
 'Datetime_month': {1: 0, 3: 1},
 'Datetime_day': {1: 0, 23: 1},
 'Datetime_hour': {9: 0, 14: 1},
 'Datetime_minute': {15: 0, 30: 1},
 'Datetime_second': {0: 0},
 'Datetime_millisecond': {0: 0}}

In [156]:
datetime_test._datetime_discretized_subcols_names

[('Duration', 0),
 ('Date_year', 2),
 ('Date_month', 3),
 ('Date_day', 4),
 ('Date_hour', 5),
 ('Date_minute', 6),
 ('Date_second', 7),
 ('Date_millisecond', 8),
 ('Duration', 0),
 ('Time_hour', 9),
 ('Time_minute', 10),
 ('Time_second', 11),
 ('Time_millisecond', 12),
 ('Duration', 0),
 ('Datetime_year', 13),
 ('Datetime_month', 14),
 ('Datetime_day', 15),
 ('Datetime_hour', 16),
 ('Datetime_minute', 17),
 ('Datetime_second', 18),
 ('Datetime_millisecond', 19),
 ('Duration', 0)]

In [161]:
# decode_numerical_discrete(df_pl:pl.DataFrame, decode_map:dict[dict[int, int]], discrete_cols:list[str])
df_decoded = decode_numerical_discrete(datetime_test._table, datetime_test.datetime_discretized_encoding_maps, [item[0] for item in datetime_test._datetime_discretized_subcols_names])

In [162]:
decode_datetime(
    df_decoded,
    [item[0] for item in datetime_test._datetime_columns],
    datetime_test.datetime_encoding_map
)

Duration,bool_test,Date,Time,Datetime
i64,i8,datetime[μs],time,datetime[μs]
null,1,null,null,null
null,0,null,null,null


In [163]:
df_pl

Date,Time,Datetime,Duration,bool_test
date,time,datetime[ns],duration[ns],bool
2026-03-23,14:30:00,2026-03-23 14:30:00,2d 5h,true
2026-01-01,09:15:00,2026-01-01 09:15:00,12h 30m,false


In [164]:
insurance_table_test._incompatible_columns

[]

In [165]:
beijing_data_test._incompatible_columns

[]

In [166]:
['a'] + ['b', 'c']

['a', 'b', 'c']

In [167]:
date_time_col_prefixes = [item[0] for item in datetime_test._datetime_columns + datetime_test._duration_columns]
date_time_col_prefixes

['Date', 'Time', 'Datetime', 'Duration']

In [168]:
date_time_subcols = []

for col_name_prefix in date_time_col_prefixes:

    curr_prefix_subcols = [item for item in datetime_test._table.columns if item.startswith(f"{col_name_prefix}_")]

    date_time_subcols = date_time_subcols + curr_prefix_subcols

In [169]:
date_time_subcols

['Date_year',
 'Date_month',
 'Date_day',
 'Date_hour',
 'Date_minute',
 'Date_second',
 'Date_millisecond',
 'Time_hour',
 'Time_minute',
 'Time_second',
 'Time_millisecond',
 'Datetime_year',
 'Datetime_month',
 'Datetime_day',
 'Datetime_hour',
 'Datetime_minute',
 'Datetime_second',
 'Datetime_millisecond']

In [170]:
datetime_test.datetime_discretized_encoding_maps

{'Duration': {45000: 0, 190800: 1},
 'Date_year': {2026: 0},
 'Date_month': {1: 0, 3: 1},
 'Date_day': {1: 0, 23: 1},
 'Date_hour': {0: 0},
 'Date_minute': {0: 0},
 'Date_second': {0: 0},
 'Date_millisecond': {0: 0},
 'Time_hour': {9: 0, 14: 1},
 'Time_minute': {15: 0, 30: 1},
 'Time_second': {0: 0},
 'Time_millisecond': {0: 0},
 'Datetime_year': {2026: 0},
 'Datetime_month': {1: 0, 3: 1},
 'Datetime_day': {1: 0, 23: 1},
 'Datetime_hour': {9: 0, 14: 1},
 'Datetime_minute': {15: 0, 30: 1},
 'Datetime_second': {0: 0},
 'Datetime_millisecond': {0: 0}}

In [173]:
# Creating the sample data
data = {
    "Date": [date(2026, 3, 23), date(2026, 1, 1)],
    "Time": [time(14, 30), time(9, 15)],
    "Datetime": pd.to_datetime(["2026-03-23 14:30:00", "2026-01-01 09:15:00"]),
    "Duration": pd.to_timedelta(["2 days 5 hours", "12 hours 30 min"]),
    "bool_test": [True, False]
}

testing_datetime_types_df = pd.DataFrame(data)

In [174]:
testing_datetime_types_df

,Date,Time,Datetime,Duration,bool_test
0,2026-03-23,14:30:00,2026-03-23 14:30:00,2 days 05:00:00,True
1,2026-01-01,09:15:00,2026-01-01 09:15:00,0 days 12:30:00,False


In [175]:
datetime_test = ArgnDataset(df_pl.to_pandas(), clip_cols = True, encode_datetime_as_discrete = False)

C:\Users\rzamb\Documents\tabular_nn\src\utils\argn_encoder_decoder.py:418: UserWarning: get_n_bins() received an empty list as an argument...
  warnings.warn("get_n_bins() received an empty list as an argument...")


In [176]:
decode_datetime(
    datetime_test._table,
    [item[0] for item in datetime_test._datetime_columns],
    datetime_test.datetime_encoding_map
)

Duration,bool_test,Date,Time,Datetime
i64,i8,datetime[μs],time,datetime[μs]
190800,1,2026-03-23 00:00:00,14:30:00,2026-03-23 14:30:00
45000,0,2026-01-01 00:00:00,09:15:00,2026-01-01 09:15:00


In [177]:
datetime_test2 = ArgnDataset(df_pl.to_pandas(), clip_cols = True, encode_datetime_as_discrete = True)

In [178]:
datetime_test2._table

Duration,bool_test,Date_year,Date_month,Date_day,Date_hour,Date_minute,Date_second,Date_millisecond,Time_hour,Time_minute,Time_second,Time_millisecond,Datetime_year,Datetime_month,Datetime_day,Datetime_hour,Datetime_minute,Datetime_second,Datetime_millisecond
i64,i8,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,1,0,1,1,0,0,0,0,1,1,0,0,0,1,1,1,1,0,0
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [179]:
#datetime_test2._datetime_discretized_subcols_names

In [180]:
datetime_test._table

Duration,bool_test,Date_year,Date_month,Date_day,Date_hour,Date_minute,Date_second,Date_millisecond,Time_hour,Time_minute,Time_second,Time_millisecond,Datetime_year,Datetime_month,Datetime_day,Datetime_hour,Datetime_minute,Datetime_second,Datetime_millisecond
i64,i8,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32,i32
190800,1,2026,3,23,0,0,0,0,14,30,0,0,2026,3,23,14,30,0,0
45000,0,2026,1,1,0,0,0,0,9,15,0,0,2026,1,1,9,15,0,0


In [181]:
decode_numerical_discrete(datetime_test2._table, datetime_test2.datetime_discretized_decoding_maps, [item[0] for item in datetime_test2._datetime_discretized_subcols_names])

Duration,bool_test,Date_year,Date_month,Date_day,Date_hour,Date_minute,Date_second,Date_millisecond,Time_hour,Time_minute,Time_second,Time_millisecond,Datetime_year,Datetime_month,Datetime_day,Datetime_hour,Datetime_minute,Datetime_second,Datetime_millisecond
i64,i8,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
null,1,2026,3,23,0,0,0,0,14,30,0,0,2026,3,23,14,30,0,0
null,0,2026,1,1,0,0,0,0,9,15,0,0,2026,1,1,9,15,0,0


In [182]:
decode_datetime(
    decode_numerical_discrete(datetime_test2._table, datetime_test2.datetime_discretized_decoding_maps, [item[0] for item in datetime_test2._datetime_discretized_subcols_names]),
    [item[0] for item in datetime_test2._datetime_columns],
    datetime_test.datetime_encoding_map
)

Duration,bool_test,Date,Time,Datetime
i64,i8,datetime[μs],time,datetime[μs]
null,1,2026-03-23 00:00:00,14:30:00,2026-03-23 14:30:00
null,0,2026-01-01 00:00:00,09:15:00,2026-01-01 09:15:00


In [183]:
testing_datetime_types_df

,Date,Time,Datetime,Duration,bool_test
0,2026-03-23,14:30:00,2026-03-23 14:30:00,2 days 05:00:00,True
1,2026-01-01,09:15:00,2026-01-01 09:15:00,0 days 12:30:00,False


In [184]:
datetime_test2.datetime_discretized_decoding_maps

{'Duration': {0: 45000, 1: 190800},
 'Date_year': {0: 2026},
 'Date_month': {0: 1, 1: 3},
 'Date_day': {0: 1, 1: 23},
 'Date_hour': {0: 0},
 'Date_minute': {0: 0},
 'Date_second': {0: 0},
 'Date_millisecond': {0: 0},
 'Time_hour': {0: 9, 1: 14},
 'Time_minute': {0: 15, 1: 30},
 'Time_second': {0: 0},
 'Time_millisecond': {0: 0},
 'Datetime_year': {0: 2026},
 'Datetime_month': {0: 1, 1: 3},
 'Datetime_day': {0: 1, 1: 23},
 'Datetime_hour': {0: 9, 1: 14},
 'Datetime_minute': {0: 15, 1: 30},
 'Datetime_second': {0: 0},
 'Datetime_millisecond': {0: 0}}